In [ ]:
# Three independent environments are needed, here we only present the main results that only relies on the "mace" environment.
# Code for three environments: 
# env: mace
# conda create -n mace python=3.12
# pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118
# python -m pip install --upgrade pip
# pip install mace-torch

# env: maml (for anchor selection)
# conda create -n maml
# pip install maml

# env: similarity (for embeddings metrics)
# git clone https://github.com/nacloos/similarity-repository.git 
# cd similarity-repository 
# pip install -e .

### Extract embeddings from different models

MACE

In [ ]:
import pandas as pd
from ase.io import read
import numpy as np
from mace.calculators import MACECalculator, mace_mp
import torch

train = read("train.xyz", index=":")

calculator = mace_mp(model='mace/models/mp0a/model_advance_cpu.model', device='cpu')
all_descriptors_mpa = []
all_atoms_number = []
#convert ase atoms to cif
for i, atoms in enumerate(train):
    atoms.write("cif_file.cif")
    init_conf=read('cif_file.cif', format='cif')
    all_atoms_number.append(init_conf.get_atomic_numbers())
    descriptors = calculator.get_descriptors(init_conf, num_layers=1)
    all_descriptors_mpa.append(descriptors)

Orb

In [ ]:
## 
from io import StringIO
import pandas as pd
df_train = pd.read_csv('mace/mp-20/train.csv')
print(len(df_train))

import torch  
from orb_models.forcefield import pretrained, atomic_system  
from ase.build import bulk 
from ase.io import read 
  
# Load a pretrained model  
device = "cuda"  # or "cuda"  
model = pretrained.orb_v3_direct_inf_omat(  
    device=device,  
    precision="float32-high"  
)  
  
all_descriptors_mpa = []
all_atoms_number = []
#convert ase atoms to cif
with torch.no_grad():  
# Get the MoleculeGNS model (the core GNN)  
    gns_model = model.model  
    for cif in df_train["cif"]:
        init_conf=read(StringIO(cif), format='cif')
        all_atoms_number.append(init_conf.get_atomic_numbers())
        graph = atomic_system.ase_atoms_to_atom_graphs(init_conf, model.system_config, device=device)  

        # 3. Final embeddings (after message passing)  
        result = gns_model(graph)  
        final_node_embeddings = result["node_features"] 
        all_descriptors_mpa.append(final_node_embeddings.cpu().numpy()) 
        #final_edge_embeddings = result["edge_features"] 
    f.close()

import pickle
with open('orb_atomnumbers_direct.pkl','wb') as f:
    pickle.dump(all_atoms_number, f)
with open('orb_embeddings_direct.pkl','wb') as f:
    pickle.dump(all_descriptors_mpa, f)

Seven

In [ ]:
#Conver the cif to graph using SevenNet
import ase.io  
from sevenn.util import unlabeled_atoms_to_input  
from sevenn.atom_graph_data import AtomGraphData 
import pandas as pd 

df_train = pd.read_csv('data/mp20/train.csv')
print(len(df_train))
all_graph = []
for cif in df_train["cif"]:
    with open('cif_file', 'w') as f:
        f.write(cif)
    # Read CIF file using ASE  
    atoms = ase.io.read('cif_file', format='cif')  
    # Convert to SevenNet input format  
    cutoff = 5.0  # Use the same cutoff as your model  
    graph_data = unlabeled_atoms_to_input(atoms, cutoff)  
    all_graph.append(graph_data)
import pickle
with open('seven_all_graph.pkl', 'wb') as f:
    pickle.dump(all_graph, f)


In [ ]:
import torch  
from typing import Dict, List  
from sevenn.util import model_from_checkpoint  
import sevenn._keys as KEY  
from tqdm import tqdm
  
class FinalEmbeddingExtractor:  
    """Extract final node embeddings before energy prediction"""  
      
    def __init__(self, model_path: str, modal: str = None):  
        """  
        Args:  
            model_path: Path to SevenNet checkpoint or model name (e.g., '7net-omat')  
            modal: Modal for multi-fidelity models (e.g., 'omat24', 'mpa')  
        """  
        self.model, self.config = model_from_checkpoint(model_path)  
        self.model.eval()  
        self.embeddings = {}  
        self.hooks = []  
          
        # Set modal if specified  
        if modal and self.model.modal_map:  
            self.modal = modal  
        else:  
            self.modal = None  
              
        self._register_final_embedding_hook()  
      
    def _register_final_embedding_hook(self):  
        """Register hook to capture embeddings before energy prediction"""  
        # Find the layer just before energy prediction  
        layer_names = list(self.model._modules.keys())  
          
        # Look for readout layers  
        readout_layers = ['reduce_input_to_hidden', 'readout_FCN']  
        target_layer = 'reduce_input_to_hidden'  
          
        for readout_layer in readout_layers:  
            if readout_layer in layer_names:  
                # Get the layer just before this readout layer  
                idx = layer_names.index(readout_layer)  
                if idx > 0:  
                    target_layer = layer_names[idx - 1]  
                break  
          
        if target_layer is None:  
            # Fallback: use the last convolution-related layer  
            for name in reversed(layer_names):  
                if 'equivariant_gate' in name or 'convolution' in name:  
                    target_layer = name  
                    break  
          
        if target_layer:
            def hook_fn(module, input, output):
                #print(f"[Hook Triggered] Layer: {module._get_name()}")
                #print(f"Output type: {type(output)}")
                # print(f"Attributes: {dir(output)}")

                # Try common field names
                if hasattr(output, 'node_feature'):
                    emb = output.node_feature
                    self.embeddings['final_embeddings'] = emb.clone().detach()
                    #print("[Hook] Captured final embeddings from node_feature.")
                elif hasattr(output, 'x'):  # Sometimes used
                    emb = output.x
                    self.embeddings['final_embeddings'] = emb.clone().detach()
                    #print("[Hook] Captured final embeddings from x.")
                else:
                    #print("[Hook] No suitable attribute found for node features.")
                    pass

            hook = self.model._modules[target_layer].register_forward_hook(hook_fn)
            self.hooks.append(hook)
            #print(f"Registered hook at layer: {target_layer}")
        else:
            raise ValueError("Could not find appropriate layer for final embeddings") 
      
    def extract_final_embeddings(self, graph_data_list: List) -> List[torch.Tensor]:
        """Extract embeddings without running force calculation"""  
        results = {'final_embeddings': [], 'atom_types': []}  
        """Extract final embeddings without gradient computation"""   
        self.model.eval()
        self.model.set_is_batch_data(False)  

        for i, graph_data in enumerate(tqdm(graph_data_list, desc="Extracting embeddings")):
            self.embeddings.clear()  
            if graph_data is None:
                results['final_embeddings'].append(None)
                results['atom_types'].append(None)
                continue
            
            with torch.no_grad():  
                # Run forward pass but stop before force calculation  
                data = self.model._preprocess(graph_data)  
                
                # Process through each module except force_output  
                for name, module in self.model._modules.items():  
                    if name == 'force_output':  
                        break  # Stop before force calculation  
                    data = module(data)  
                
            emb = self.embeddings.get('final_embeddings', None)
            results['final_embeddings'].append(emb)

            atomic_numbers = graph_data[KEY.ATOMIC_NUMBERS]
            atomic_numbers = atomic_numbers.cpu().tolist() if torch.is_tensor(atomic_numbers) else list(atomic_numbers)
            results['atom_types'].append(atomic_numbers)
            if i == 0:
                print(f"\n[Debug] First embedding (layer: {layer_name})")
                print(f"  Shape: {emb.shape}")
                print(f"  Dtype: {emb.dtype}")
                print(f"  Sample values:\n{emb[:1]}")
                 
        return results
      
    def cleanup(self):  
        """Remove hooks"""  
        for hook in self.hooks:  
            hook.remove()  
        self.hooks.clear()  
  
# Usage function  
def extract_final_embeddings_from_graphs(graph_data_list: List,   
                                        model_name: str = '7net-omat',  
                                        modal: str = None) -> List[torch.Tensor]:  
    """  
    Convenience function to extract final embeddings  
      
    Args:  
        graph_data_list: List of processed AtomGraphData objects  
        model_name: SevenNet model name or path  
        modal: Modal for multi-fidelity models  
          
    Returns:  
        List of final embedding tensors  
    """  
    extractor = FinalEmbeddingExtractor(model_name, modal)  
      
    try:  
        embeddings = extractor.extract_final_embeddings(graph_data_list)  
        return embeddings  
    finally:  
        extractor.cleanup()  
  
# Example usage  
if __name__ == "__main__":  
    # Assuming you have your graph_data_list ready  
    # graph_data_list = [your processed AtomGraphData objects]  
      
    # Extract final embeddings  
    final_embeddings = extract_final_embeddings_from_graphs(  
        all_graph,   
        model_name='7net-omat'  # or '7net-mf-ompa' with modal='omat24'  
    )  
      
    # Print shapes of extracted embeddings  
    for i, embedding in enumerate(final_embeddings):  
        if embedding is not None:  
            print(f"Structure {i}: Final embedding shape {embedding.shape}")  
        else:  
            print(f"Structure {i}: No embedding extracted")

### Anchor selection

In [ ]:
from __future__ import annotations

import pickle

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

from maml.sampling.direct import BirchClustering, DIRECTSampler, SelectKFromClusters

In [ ]:
with open('small_all_atom_embeddings.pkl', 'rb') as f: 
    all_atom_embeddings = pickle.load(f)

In [ ]:
DIRECT_sampler = DIRECTSampler(
    structure_encoder=None,
    clustering=BirchClustering(n=100, threshold_init=0.3),
    select_k_from_clusters=SelectKFromClusters(k=1),
)

In [ ]:
DIRECT_selection = DIRECT_sampler.fit_transform(all_atom_embeddings)
print(
    f"DIRECT selected {len(DIRECT_selection['selected_indexes'])} structures from "
    f"{len(DIRECT_selection['PCAfeatures'])} structures in MPF.2021.2.8.All."
)

In [ ]:
## save the selected samples
import pickle
selected_indexes = DIRECT_selection["selected_indexes"]
selected_anchors = [all_atom_embeddings[i] for i in selected_indexes]
#selected_anchors = [chem_feature_data[i] for i in selected_indexes]
with open('mace_small_anchor_100.pkl', 'wb') as f: 
    pickle.dump(selected_anchors, f)

### Example anchor indices, if you get all the embeddings from mp_20 train.csv

In [ ]:
# with this indices, you can get the selected anchors from your full embeddings, and use them as the probe anchors to test your model embeddings's distrubtuion
anchor_indices = [222430, 144721, 282484, 38074, 187477, 19816, 38484, 204870, 223370, 180858, 239896, 271612, 270370, 256655, 164, 108486, 282577, 168218, 155521, 236527, 139439, 134383, 6039, 30614, 181679, 25264, 185297, 125740, 75455, 195535, 281120, 158363, 252468, 102981, 190032, 107059, 226948, 44342, 236453, 131321, 205532, 174744, 169073, 104658, 230, 9999, 17943, 279479, 225757, 34751, 2456, 1932, 25170, 52622, 222079, 227723, 164773, 135893, 12023, 258104, 146125, 162523, 188573, 187378, 113021, 63466, 260179, 214, 93285, 277786, 216748, 248850, 237493, 114495, 7366, 139062, 218251, 143350, 169531, 88557, 211490, 169214, 100378, 270997, 84444, 241854, 154502, 246833, 112772, 160036, 117303, 182341, 119695, 37694, 193103, 261515, 97870, 264644, 107630, 196220]

### Transformation

In [ ]:
### use anchor to project the embeddings
### Select anchors
import torch  
import torch.nn.functional as F 

def calculate_relative_embeddings(embeddings: torch.Tensor, anchors):  
    """  
    Calculate relative embeddings using the first num_anchors embeddings as anchors.  
      
    Args:  
        embeddings: Your embeddings tensor [num_samples, embedding_dim]  
        num_anchors: Number of anchors to use (default: 40)  
      
    Returns:  
        Relative embeddings tensor [num_samples, num_anchors]  
    """  
    # Simple relative projection using cosine similarity (from demo)  
    x_normalized = F.normalize(embeddings, p=2, dim=-1)  
    anchors_normalized = F.normalize(anchors, p=2, dim=-1)  
    relative_embeddings = torch.einsum("bm, am -> ba", x_normalized, anchors_normalized)  
      
    return relative_embeddings 

Platonic_rep = calculate_relative_embeddings(torch.tensor(all_atom_embeddings, dtype=torch.float64), torch.tensor(selected_anchors, dtype=torch.float64))


### Compare models

Optimal Transport (Package: https://pythonot.github.io/contributing.html)

In [ ]:
## calculate and plot the optimal transport plan between the embeddings of the two models
import ot
import numpy as np
def calculate_ot_plan(
    embeddings1,
    embeddings2,
    metric='euclidean',
    weights1=None,
    weights2=None,
    use_sinkhorn=False,
    reg=1e-3
):
    embeddings1 = np.asarray(embeddings1)
    embeddings2 = np.asarray(embeddings2)

    cost_matrix = ot.dist(embeddings1, embeddings2, metric=metric)

    n, m = len(embeddings1), len(embeddings2)
    a = np.asarray(weights1) if weights1 is not None else np.ones(n) / n
    b = np.asarray(weights2) if weights2 is not None else np.ones(m) / m

    if use_sinkhorn:
        ot_plan = ot.sinkhorn(a, b, cost_matrix, reg)
    else:
        ot_plan = ot.emd(a, b, cost_matrix)
    total_cost = np.sum(ot_plan * cost_matrix)
    print("Total OT cost:", total_cost)
    print("Sum of OT plan:", np.sum(ot_plan))
    return ot_plan, total_cost

Mutual KNN, CKA, and Procrustes (Package: https://github.com/nacloos/similarity-repository/tree/main)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import similarity

knn_score = similarity.make("measure/platonic/mutual_knn-topk={topk}")
#cka_score = similarity.make("measure/rtd/cka-kernel=linear-hsic=gretton-score")
#procrustes_score = similarity.make("measure/netrep/procrustes-distance=angular")

filtered_embeddings_list = [np.array(m) for m in embeddings]

n = len(filtered_embeddings_list)
score_matrix = np.zeros((n, n))
import torch
for i in range(n):
    for j in range(n):
        if i == j:
            score_matrix[i, j] = 1.0 # Diagonal elements
        elif i < j:
            torch.manual_seed(42)
            tensor_i = filtered_embeddings_list[i]
            tensor_j = filtered_embeddings_list[j]

            indices = torch.randperm(tensor_i.shape[0])[:1000]
            i_samples = tensor_i[indices]
            j_samples = tensor_j[indices]
            score_matrix[i, j] = knn_score(i_samples, j_samples)
        else:
            score_matrix[i, j] = score_matrix[j, i]  # Symmetric matrix



### Space group

In [ ]:
#read csv and get the indices
import pandas as pd
import numpy as np
found_df = pd.read_csv('found_indices_spacegroup221new.csv')
found_start221 = found_df['start'].tolist()
found_end221 = found_df['end'].tolist()
found_id221 = found_df['material_id'].tolist()
found_id221_flat = []
for item in found_id221:
    found_id221_flat.extend(item.strip('[]').replace("'", "").split(', '))
print(len(found_id221_flat))
found_com = found_df['composition'].tolist()

In [ ]:
embeddings_221_o = []
embeddings_221_p = []
for j in range(8): 
    model_o = []
    model_p = []
    for i in range(len(found_start221)):
        embeddings_225_os = np.array(all_embeddings[j][found_start221[i]:found_end221[i]])
        embeddings_225_ps = np.array(platonic_rep[j][found_start221[i]:found_end221[i]])
        model_o.append(embeddings_225_os)
        model_p.append(embeddings_225_ps)
#flatten the list
    model_o = [item for sublist in model_o for item in sublist]
    model_p = [item for sublist in model_p for item in sublist]
    embeddings_221_o.append(model_o)
    embeddings_221_p.append(model_p)
print(len(embeddings_221_o[1]))

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression

def estimate_intrinsic_dimension_fixed(X, k_omit=1):
    """
    Estimates the Intrinsic Dimension (ID) using the Two-NN algorithm (Facco E, et al. 2017).
    Includes fix for -ln(0) error by truncating the distribution tail.

    Args:
        X: numpy array of shape (n_samples, n_features) (your embeddings).
        k_omit: Number of points to omit from the end of the sorted mu distribution.
                Must be at least 1 to prevent log(0) error.

    Returns:
        float: The estimated intrinsic dimension.
    """
    N = X.shape[0]

    # 1. Compute distances to the first 2 nearest neighbors
    nbrs = NearestNeighbors(n_neighbors=3, algorithm='auto').fit(X)
    distances, _ = nbrs.kneighbors(X)

    # Extract r1 (dist to 1st neighbor) and r2 (dist to 2nd neighbor)
    r1 = distances[:, 1]
    r2 = distances[:, 2]
    num_collapsed1 = np.sum(r1 == 0)
    num_total1 = len(r1)
    #record the indice of collapsed points
    collapsed_indices = np.where(r1 == 0)[0]

    # print("Collapse fraction r1:", num_collapsed1 / num_total1)
    # num_collapsed = np.sum(r2 == 0)
    # num_total = len(r2)
    # print("Collapse fraction r2:", num_collapsed / num_total)
    # # print if r2 not greater than r1
    # num_invalid = np.sum(r2 < r1)
    # print("Number of invalid r2 <= r1:", num_invalid)

    # from sklearn.cluster import DBSCAN
    # labels = DBSCAN(eps=1e-12, min_samples=2).fit_predict(X) 
    # print("Number of unique clusters (including noise):", len(set(labels)))  

    # # Filter out cases where r1 is 0 (duplicates)
    mask = r1 > 0
    r1 = r1[mask]
    r2 = r2[mask]

    # 2. Compute the ratios mu = r2 / r1
    mu = r2 / r1


    # 3. Compute the empirical cumulate distribution
    mu_sorted = np.sort(mu)
    F_mu = np.arange(1, len(mu_sorted) + 1) / len(mu_sorted)

    # --- FIX APPLIED HERE ---
    # Truncate arrays to omit k_omit points (default is the last point)
    mu_trunc = mu_sorted[:-k_omit]
    F_mu_trunc = F_mu[:-k_omit]

    # Check if there's enough data left for regression
    if len(mu_trunc) < 2:
        return np.nan # Not enough data after filtering

    # 4. Linear Regression on the log-coordinates
    x = np.log(mu_trunc).reshape(-1, 1)
    y = -np.log(1 - F_mu_trunc).reshape(-1, 1)

    # Fit line through origin (fit_intercept=False is crucial)
    reg = LinearRegression(fit_intercept=False).fit(x, y)

    estimated_id = reg.coef_[0][0]

    # # --- Claude ---
    # log_mu = np.log(mu_sorted + 1e-10)  # Add small constant to avoid log(0)
    # mean_log_mu = np.mean(log_mu)
    # estimated_id = 1 / mean_log_mu

    #return estimated_id
    return num_collapsed1 / num_total1, collapsed_indices, estimated_id


original_ids_221 = []
platonic_ids_221 = []
collaped_o_221 = []
collaped_p_221 = []
# 3. Run the Test
for model in range(len(embeddings_221_o)):
    id_A, collapsed_indices_a, _ = estimate_intrinsic_dimension_fixed(np.array(embeddings_221_o[model]))
    original_ids_221.append(id_A)
    collaped_o_221.append(collapsed_indices_a)
    print(f"--- Intrinsic Dimension Results ---")
    print(f"Model A (Preserves Manifold): ID ≈ {id_A:.2f}")
    id_B, collapsed_indices_b, _  = estimate_intrinsic_dimension_fixed(np.array(embeddings_221_p[model]))
    platonic_ids_221.append(id_B)
    collaped_p_221.append(collapsed_indices_b)
    print(f"Platonic Model A (Preserves Manifold): ID ≈ {id_B:.2f}")